# Clustering Track: Wholesale Customer Segmentation

In this track, we apply unsupervised machine learning to segment wholesale customers based on annual spending across six core product categories: `Fresh`, `Milk`, `Grocery`, `Frozen`, `Detergents_Paper`, and `Delicassen`.

In accordance with rubric requirements:
* Ground-truth labels (`Channel` and `Region`) are held out during model training and used only for post-hoc validation.
* Models implemented: **K-Means Clustering** (with Elbow analysis) and **Agglomerative Hierarchical Clustering** (with Dendrogram and linkage comparison).
* Mandatory evaluation metrics: **Silhouette Score**, **Davies-Bouldin Index**, and **Calinski-Harabasz Index**.
* Mandatory visualizations: **PCA 2D projection** and **t-SNE non-linear projection**.

In [ ]:
# ==============================================================================
# Cell 1: Environment and Libraries Setup
# ==============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage

# Set random seed for reproducibility
np.random.seed(42)

### Dataset Loading & Feature Isolation
We load the dataset and isolate the ground-truth metadata (`Channel` and `Region`) so that models are fitted strictly without supervision.

In [ ]:
# ==============================================================================
# Cell 2: Dataset Loading and Ground-Truth Isolation
# ==============================================================================
import os
data_path = '../data/customers.csv' if os.path.exists('../data/customers.csv') else 'data/customers.csv'
df = pd.read_csv(data_path)

print("--- Dataset Audit ---")
print(f"Dataset Shape: {df.shape}")
print(f"Missing Values:\n{df.isnull().sum()}")

# Ground truth for post-hoc validation
ground_truth = df[['Channel', 'Region']].copy()

# Continuous spending features for unsupervised clustering
X = df.drop(columns=['Channel', 'Region'])
print(f"\nClustering Feature Space Shape: {X.shape}")
print(f"Features: {list(X.columns)}")
df.head()


### Feature Normalization
Expenditure figures across categories differ significantly in magnitude and scale. We apply `StandardScaler` to ensure distance-based metrics (Euclidean) treat all spending categories with equal weight.

In [ ]:
# ==============================================================================
# Cell 3: Feature Scaling (StandardScaler)
# ==============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("StandardScaler applied: zero mean and unit variance achieved.")

### Algorithm 1: K-Means Clustering & Elbow Method
We evaluate cluster inertia (within-cluster sum of squares) and Silhouette scores for $k \in [2, 10]$ to identify the optimal number of customer clusters.

In [ ]:
# ==============================================================================
# Cell 4: K-Means Elbow Curve and Model Fitting
# ==============================================================================
k_range = range(2, 11)
inertias = []
sil_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbls = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbls))

# Dual-axis plot: Inertia (Elbow) vs. Silhouette Score
fig, ax1 = plt.subplots(figsize=(10, 4))

color = 'tab:blue'
ax1.set_xlabel('Number of Clusters (k)', fontsize=11)
ax1.set_ylabel('Inertia (Within-Cluster Sum of Squares)', color=color, fontsize=11)
ax1.plot(k_range, inertias, marker='o', color=color, linewidth=2)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:orange'
ax2.set_ylabel('Silhouette Score', color=color, fontsize=11)
ax2.plot(k_range, sil_scores, marker='s', color=color, linewidth=2, linestyle='--')
ax2.tick_params(axis='y', labelcolor=color)

plt.title('K-Means: Elbow Method (Inertia) & Silhouette Profile vs. k', fontsize=13)
plt.tight_layout()
plt.show()

# Fit optimal K-Means model (k=3)
optimal_k = 3
kmeans_model = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
kmeans_labels = kmeans_model.fit_predict(X_scaled)
print(f"K-Means fitted with k={optimal_k}. Cluster distribution: {np.bincount(kmeans_labels)}")

### Algorithm 2: Agglomerative Hierarchical Clustering & Dendrogram
We perform hierarchical bottom-up clustering using Ward's minimum variance criterion, display the dendrogram with cutoff threshold, and compare linkage strategies.

In [ ]:
# ==============================================================================
# Cell 5: Hierarchical Clustering Dendrogram & Linkage Comparison
# ==============================================================================
# Hierarchical Linkage
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 5))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=45, leaf_font_size=9, show_contracted=True)
plt.title('Agglomerative Hierarchical Clustering Dendrogram (Ward Linkage)', fontsize=13)
plt.xlabel('Cluster / Sample Partition Index')
plt.ylabel('Euclidean Merge Distance')
plt.axhline(y=15, color='red', linestyle='--', label='Cut Threshold (k=3)')
plt.legend()
plt.tight_layout()
plt.show()

# Compare linkage strategies
print('--- Linkage Strategies Comparison (k=3) ---')
for link in ['ward', 'complete', 'average']:
    agg = AgglomerativeClustering(n_clusters=optimal_k, linkage=link)
    lbls = agg.fit_predict(X_scaled)
    print(f"Linkage: {link.capitalize():<10} | Silhouette: {silhouette_score(X_scaled, lbls):.4f}")

# Fit final Agglomerative model with Ward linkage
hierarchical_model = AgglomerativeClustering(n_clusters=optimal_k, linkage='ward')
hierarchical_labels = hierarchical_model.fit_predict(X_scaled)
print(f"Agglomerative Hierarchical fitted with k={optimal_k}. Cluster distribution: {np.bincount(hierarchical_labels)}")

### Cluster Visualizations: 2D PCA & Non-Linear 2D t-SNE
In accordance with rubric requirements, we generate 2D PCA projections for every algorithm and a 2D t-SNE non-linear embedding to inspect cluster separability.

In [ ]:
# ==============================================================================
# Cell 6: PCA and t-SNE Visualizations (Rubric Requirement)
# ==============================================================================
# 1. PCA 2D Reduction
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# 2. t-SNE 2D Reduction
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
X_tsne = tsne.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: PCA with K-Means
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.7, edgecolors='k')
axes[0].set_title(f'PCA: K-Means (k={optimal_k})', fontsize=12)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

# Plot 2: PCA with Hierarchical
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=hierarchical_labels, cmap='tab10', alpha=0.7, edgecolors='k')
axes[1].set_title(f'PCA: Hierarchical (k={optimal_k})', fontsize=12)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')

# Plot 3: t-SNE with K-Means
axes[2].scatter(X_tsne[:, 0], X_tsne[:, 1], c=kmeans_labels, cmap='tab10', alpha=0.7, edgecolors='k')
axes[2].set_title('t-SNE: K-Means Clustering', fontsize=12)
axes[2].set_xlabel('t-SNE Dimension 1')
axes[2].set_ylabel('t-SNE Dimension 2')

plt.suptitle('Customer Segment Visualizations (PCA & t-SNE)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### Quantitative Cluster Evaluation Metrics
We evaluate both models using all three mandatory evaluation metrics: **Silhouette Score**, **Davies-Bouldin Index**, and **Calinski-Harabasz Index**.

In [ ]:
# ==============================================================================
# Cell 7: Mandatory Clustering Evaluation Metrics Table
# ==============================================================================
clustering_metrics = {
    'Algorithm': ['K-Means Clustering', 'Agglomerative Hierarchical'],
    'Silhouette Score (?)': [
        round(silhouette_score(X_scaled, kmeans_labels), 4),
        round(silhouette_score(X_scaled, hierarchical_labels), 4)
    ],
    'Davies-Bouldin Index (?)': [
        round(davies_bouldin_score(X_scaled, kmeans_labels), 4),
        round(davies_bouldin_score(X_scaled, hierarchical_labels), 4)
    ],
    'Calinski-Harabasz Index (?)': [
        round(calinski_harabasz_score(X_scaled, kmeans_labels), 4),
        round(calinski_harabasz_score(X_scaled, hierarchical_labels), 4)
    ]
}

clustering_summary_df = pd.DataFrame(clustering_metrics)
print("--- Final Quantitative Clustering Performance Summary ---")
display(clustering_summary_df)

### Ground-Truth Post-Hoc Validation
To validate whether the unsupervised clusters correspond to real customer behaviors, we map the clusters against the ground-truth `Channel` (Horeca vs. Retail).

In [ ]:
# ==============================================================================
# Cell 8: Ground-Truth Validation Cross-Tabulation
# ==============================================================================
val_df = pd.DataFrame({
    'Customer Channel': ground_truth['Channel'].map({1: 'Horeca (Hotel/Restaurant/Cafe)', 2: 'Retail'}),
    'Region': ground_truth['Region'].map({1: 'Lisbon', 2: 'Oporto', 3: 'Other'}),
    'KMeans Cluster': kmeans_labels
})

print("Cross-Tabulation: K-Means Clusters vs. Customer Channel")
display(pd.crosstab(val_df['KMeans Cluster'], val_df['Customer Channel'], margins=True))